# 02 Numerical Preliminaries

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

## The Lens: The Illusion of Continuity

**What problem are we solving?**  
Economic theory deals with continuous variables: prices, quantities, probabilities. We write models using real numbers ($\mathbb{R}$). But computers are discrete machines. They cannot represent the continuum. They approximate real numbers using **floating-point arithmetic**, which introduces small but cumulative errors.

**Why this method?**  
Understanding the limitations of the machine is the first step in computational economics.
*   **Machine Epsilon:** The smallest difference the computer can distinguish.
*   **Round-off Error:** The error introduced by finite precision.
*   **Truncation Error:** The error introduced by approximating an infinite process (like a limit) with a finite one.

This notebook is about "knowing your tools." Before we solve complex models, we must understand how the computer counts.

## Learning Objectives

By the end of this notebook, you will be able to:
1.  **Explain** how computers represent numbers using IEEE 754 standard.
2.  **Quantify** numerical errors (Absolute, Relative, Round-off, Truncation).
3.  **Diagnose** stability and conditioning issues in algorithms.
4.  **Implement** arbitrary-precision arithmetic when necessary.
5.  **Analyze** the algorithmic complexity (Big-O) of your code.

## Prerequisites

*   **01-Foundations/12_NumPy.ipynb**: Basic NumPy usage.

In [ ]:
# === Environment Setup ===
import sys
import struct
from decimal import Decimal, getcontext
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import hilbert

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'figure.dpi': 120})
np.set_printoptions(suppress=True, linewidth=120, precision=8)
getcontext().prec = 50 # Set precision for Decimal objects

## 1. How Computers Represent Numbers

### 1.1 IEEE 754 Floating-Point Representation

A standard 64-bit float (`float64` in NumPy) consists of:
1.  **Sign Bit (1 bit):** Positive or negative.
2.  **Exponent (11 bits):** Magnitude range.
3.  **Mantissa (52 bits):** Precision (significant digits).

Value = $(-1)^{\text{sign}} \times (1 + \text{mantissa}) \times 2^{\text{exponent} - 1023}$

**Consequence:** Numbers like `0.1` cannot be represented exactly in binary, leading to `0.1 + 0.2 != 0.3`.

In [ ]:
def float_to_bits(f):
    """Unpacks a 64-bit float into its sign, exponent, and mantissa bits."""
    packed = struct.pack('!d', f)
    integers = struct.unpack('!Q', packed)[0]
    # Extract bits
    sign = (integers >> 63) & 1
    exponent = (integers >> 52) & 0x7FF
    mantissa = integers & 0xFFFFFFFFFFFFF
    return sign, exponent, mantissa

s, e, m = float_to_bits(19.94)
print(f"Float: 19.94")
print(f"Sign: {s}")
print(f"Exponent: {e} (Stored)")
print(f"Mantissa: {m}")

### 1.2 Machine Epsilon

**Machine Epsilon** ($\epsilon_{mach}$) is the distance from 1.0 to the next largest representable float. For 64-bit floats, $\epsilon_{mach} \approx 2.22 \times 10^{-16}$.

**Crucially**, the spacing between floats is **not uniform**. It grows with magnitude. The gap between 1.0 and 2.0 is small; the gap between $10^{16}$ and the next number is large (> 1.0).

### 1.3 Special Values: Infinity and NaN

The IEEE 754 standard reserves specific bit patterns for non-real results:
*   **Infinity (`inf`):** Result of overflow (e.g., `1.0/0.0` or `exp(1000)`).
*   **Not a Number (`nan`):** Result of undefined operations (e.g., `0.0/0.0` or `inf - inf`). 
    *   *Warning:* `nan != nan`. Always use `np.isnan()` to check.

In [ ]:
eps = np.finfo(float).eps
print(f"Machine Epsilon: {eps:.2e}")

gap_at_1 = np.nextafter(1.0, 2.0) - 1.0
gap_at_large = np.nextafter(1e16, 2e16) - 1e16

print(f"Gap at 1.0:      {gap_at_1:.2e}")
print(f"Gap at 1e16:     {gap_at_large:.2f}")
print("Notice: At 1e16, the computer cannot represent odd integers!")

## 2. A Taxonomy of Numerical Error

1.  **Rounding Error:** Caused by finite precision (e.g., storing $\pi$ as a float).
2.  **Truncation Error:** Caused by approximating an infinite process (e.g., Taylor series, iterative limit) with a finite one.

### Absolute vs. Relative Error

*   **Absolute:** $E_{abs} = |x_{true} - x_{approx}|$
*   **Relative:** $E_{rel} = \frac{|x_{true} - x_{approx}|}{|x_{true}|}$

### Forward vs. Backward Error
This distinction is crucial for stability analysis:
*   **Forward Error:** The difference between the computed answer and the true answer ($|y_{computed} - y_{true}|$). This is what we usually care about.
*   **Backward Error:** The size of the perturbation to the *input* that would make the computed answer the *exact* answer for that perturbed input. If an algorithm has small backward error, it is "Backward Stable". It gives the right answer to a slightly wrong problem.

**Error Propagation Formula:**
If $x_{approx} = x (1 + \epsilon_x)$ and $y_{approx} = y (1 + \epsilon_y)$:
*   **Multiplication:** $x_{approx} y_{approx} \approx xy(1 + \epsilon_x + \epsilon_y)$. Relative errors add.
*   **Addition:** $x_{approx} + y_{approx} = (x+y) + x\epsilon_x + y\epsilon_y$. Relative error is $\frac{x\epsilon_x + y\epsilon_y}{x+y}$. If $x \approx -y$, the denominator is small, and error explodes (**Cancellation**).

**Best Practice:** Always use relative error (or `np.isclose` which handles both) unless the value is near zero.

## 3. Diagnosing Problems: Conditioning and Stability

### 3.1 Conditioning (The Problem)
A problem is **ill-conditioned** if small changes in inputs cause huge changes in outputs. Example: Solving $Ax=b$ with a Hilbert matrix.

### 3.2 Stability (The Algorithm)
An algorithm is **unstable** if it amplifies rounding errors. Example: **Catastrophic Cancellation** when subtracting two nearly equal numbers.

In [ ]:
# Demonstrating Catastrophic Cancellation in Variance Calculation
rng = np.random.default_rng(42)
# Large mean, small variance
x = rng.normal(1e9, 1.0, size=10000)

# Naive formula: E[x^2] - (E[x])^2
var_naive = np.mean(x**2) - np.mean(x)**2

# Stable formula (NumPy's implementation)
var_stable = np.var(x)

print(f"True Variance:   1.0 (approx)")
print(f"Stable Var:      {var_stable:.4f}")
print(f"Naive Var:       {var_naive:.4f} (Often completely wrong or negative!)")

In [ ]:
# Demonstrating Catastrophic Cancellation in Quadratic Formula
# Equation: ax^2 + bx + c = 0. Roots: (-b +/- sqrt(b^2 - 4ac)) / 2a
# If b^2 approx 4ac, the square root is close to b.
a = 1.0
b = 1.0e8
c = 1.0

# Naive Standard Formula for the smaller root (requires subtraction)
root_naive = (-b + np.sqrt(b**2 - 4*a*c)) / (2*a)

# Rationalized Formula (avoids subtraction)
root_stable = -2*c / (b + np.sqrt(b**2 - 4*a*c))

print(f"Naive Root:  {root_naive}")
print(f"Stable Root: {root_stable}")
print("Notice the loss of precision in the naive method!")

In [ ]:
# Example: Loss of Significance in Finance
# Subtracting two large, nearly identical numbers (e.g., bond prices) results in loss of precision.
price_A = 100000.000000001
price_B = 100000.000000000

# In standard 64-bit float (approx 15-17 decimal digits of precision)
diff = price_A - price_B
print(f"Difference: {diff}")
print("The result has only 1 significant digit of precision left! The other 15 were lost in the subtraction.")

### Kahan Summation Algorithm
When summing many numbers (like computing the mean of a large dataset), small errors accumulate. Kahan summation uses a separate variable to track the "compensation" for lost low-order bits.

$$ \text{sum} = (\text{sum} + \text{input}) + \text{correction} $$

This effectively carries precision into the next step.

## 4. Arbitrary-Precision Arithmetic

For finance (where every penny counts) or high-precision math, standard floats are insufficient. Python's `decimal` module mimics human arithmetic.

In [ ]:
val_float = 0.1 + 0.1 + 0.1 - 0.3
print(f"Float Math:   {val_float:.20f} (Not zero!)")

val_decimal = Decimal('0.1') + Decimal('0.1') + Decimal('0.1') - Decimal('0.3')
print(f"Decimal Math: {val_decimal} (Exact)")

## 5. Rates of Convergence

When iterating (e.g., to find a root $x^*$), how fast does the error $\epsilon_k = |x_k - x^*|$ shrink?

1.  **Linear Convergence:** $\epsilon_{k+1} \le C \epsilon_k$ with $C < 1$. The number of correct digits increases by a constant amount each step. (e.g., Bisection).
2.  **Quadratic Convergence:** $\epsilon_{k+1} \le C \epsilon_k^2$. The number of correct digits *doubles* each step. (e.g., Newton's Method).

Quadratic is much faster. If $\epsilon_k = 10^{-2}$, then $\epsilon_{k+1} \approx 10^{-4}$, $\epsilon_{k+2} \approx 10^{-8}$.

## 6. Algorithmic Complexity

**Big-O Notation** describes how runtime scales with input size $N$.

*   $O(1)$: Constant (Dictionary lookup).
*   $O(N)$: Linear (Looping through a list).
*   $O(N \log N)$: Log-Linear (Sorting, FFT). Crucial for efficient heterogeneous agent models.
*   $O(N^2)$: Quadratic (Nested loops, matrix-vector multiplication).
*   $O(N^3)$: Cubic (Matrix multiplication/inversion). Common in econometrics (OLS).

### Amortized Analysis
Some operations are expensive occasionally but cheap on average. 
*   Example: Appending to a Python list. Usually $O(1)$, but occasionally $O(N)$ when the memory buffer fills up and the list must be copied. The *amortized* cost is $O(1)$.

## Summary

**Key Takeaways:**
*   **Floats are approximations:** Always use `np.isclose`, never `==`.
*   **Conditioning Matters:** A stable algorithm cannot fix an ill-conditioned problem.
*   **Avoid Cancellation:** Rewrite formulas (like variance or the quadratic formula) to avoid subtracting nearly equal numbers.
*   **Know the Cost:** Be aware of the Big-O complexity of your operations, especially inside loops.

## Exercises

### 1. Conceptual: Harmonic Series Paradox
The harmonic series $\sum \frac{1}{n}$ diverges. Write a script to sum this series using `float32`. Does it diverge or converge? Why?

### 2. Applied: Robust Quadratic Solver
The standard quadratic formula $\frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$ is unstable when $b^2 \gg 4ac$. Implement a function `solve_quadratic(a, b, c)` that uses the alternative form or rationalization to avoid catastrophic cancellation.

### 3. Challenge: Financial Precision
Calculate the value of a $1 investment at 5% interest compounded annually for 100 years. Compare `float` vs `Decimal`. At what year do they diverge by more than 1 cent?